In [5]:
from google.colab import files

uploaded = files.upload()

Saving 05_person_skills.csv to 05_person_skills.csv
Saving 06_skills.csv to 06_skills.csv
Saving 01_people.csv to 01_people.csv
Saving 02_abilities.csv to 02_abilities.csv
Saving 03_education.csv to 03_education.csv
Saving 04_experience.csv to 04_experience.csv


In [6]:
import pandas as pd

people = pd.read_csv("01_people.csv")
abilities = pd.read_csv("02_abilities.csv")
education = pd.read_csv("03_education.csv")
experience = pd.read_csv("04_experience.csv")
person_skills = pd.read_csv("05_person_skills.csv")
skills = pd.read_csv("06_skills.csv")

print("✅ All files loaded successfully")
print(people.head())

✅ All files loaded successfully
   person_id                                               name email phone  \
0          1                             Database Administrator   NaN   NaN   
1          2                             Database Administrator   NaN   NaN   
2          3                      Oracle Database Administrator   NaN   NaN   
3          4  Amazon Redshift Administrator and ETL Develope...   NaN   NaN   
4          5             Scrum Master Scrum Master Scrum Master   NaN   NaN   

  linkedin  
0      NaN  
1      NaN  
2      NaN  
3      NaN  
4      NaN  


In [7]:
import pandas as pd
import re

In [10]:
skill_map = {skill: skill for skill in skills["skill"].dropna().tolist()}

In [9]:
print(skills.columns)
print(skills.head())

Index(['skill'], dtype='object')
                                               skill
0                                       Mongo DB-3.2
1                                          JNDI LDAP
2                                  Stored Procedures
3                            Perform ad-hoc analysis
4  Monitored and resolved flight crew legality is...


In [11]:
def clean(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [13]:
person_skills_group = person_skills.groupby('person_id')['skill'].apply(list).to_dict()
pid = people['person_id'].iloc[0]

skill_list = person_skills_group.get(pid, [])
skills_list = [skill_map.get(i, i) for i in skill_list]
skills_list = [clean(s) for s in skills_list if s]

In [ ]:
import pdfplumber
import re
import json

def extract_resume_data(pdf_path):
    full_text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            full_text += page.extract_text() + "\n"

    # ── LINKS ─────────────────────────────────────────────────────────────────
    github = list(dict.fromkeys(re.findall(
        r'(?:https?://)?(?:www\.)?github\.com/[A-Za-z0-9_.\-]+(?:/[A-Za-z0-9_.\-]+)*',
        full_text, re.I
    )))
    linkedin = list(dict.fromkeys(re.findall(
        r'(?:https?://)?(?:www\.)?linkedin\.com/in/[A-Za-z0-9_.\-]+',
        full_text, re.I
    )))

    # ── SECTION SPLITTER ──────────────────────────────────────────────────────
    HEADERS = ["Professional Summary", "Education", "Technical Skills",
               "Projects", "Leadership & Achievements", "Key Strengths",
               "Certifications", "Courses", "Licenses & Certifications"]
    header_re = re.compile(
        r'(?m)^\s*(' + '|'.join(re.escape(h) for h in HEADERS) + r')\s*$', re.I
    )

    def get_section(name):
        matches = list(header_re.finditer(full_text))
        for i, m in enumerate(matches):
            if re.match(name, m.group(1).strip(), re.I):
                start = m.end()
                end = matches[i+1].start() if i+1 < len(matches) else len(full_text)
                return full_text[start:end].strip()
        return ""

    # ── SKILLS ────────────────────────────────────────────────────────────────
    skills_raw = get_section("Technical Skills")
    skills = {}
    cat_re = re.compile(
        r'(Languages?|Core|Frameworks?\s*[&]?\s*Libraries?|Tools?\s*[&]?\s*Platforms?|Expertise)\s*:\s*', re.I
    )
    cat_matches = list(cat_re.finditer(skills_raw))
    for i, cm in enumerate(cat_matches):
        cat = cm.group(1).strip()
        start = cm.end()
        end = cat_matches[i+1].start() if i+1 < len(cat_matches) else len(skills_raw)
        items = [s.strip(" •\n\t–—") for s in re.split(r'[,•]+', skills_raw[start:end])]
        skills[cat] = [s for s in items if 2 <= len(s) <= 80 and not cat_re.match(s)]

    # ── PROJECT TITLES ONLY ───────────────────────────────────────────────────
    projects_raw = get_section("Projects")
    project_titles = []
    for line in projects_raw.splitlines():
        line = line.strip()
        # Only lines starting with a bullet
        if not re.match(r'^[•\-]', line):
            continue
        # Strip bullet
        line = re.sub(r'^[•\-]\s*', '', line).strip()
        # Take only what's before — or – (the title part)
        title = re.split(r'\s*[—–]\s*', line, maxsplit=1)[0].strip()
        # Remove trailing parenthetical like "(Winner, 8th Mile RVCE)"
        title = re.sub(r'\s*\(.*?\)\s*$', '', title).strip()
        if title and len(title) <= 100:
            project_titles.append(title)

    # ── CERTIFICATION SKILLS ──────────────────────────────────────────────────
    # Look in Achievements/Certifications section for "Certified in X" patterns
    ach_raw = get_section("Leadership & Achievements")
    cert_raw = get_section("Certifications") or get_section(r"Licenses?\s*&?\s*Certifications?")

    combined_cert_text = ach_raw + "\n" + cert_raw

    certification_skills = []
    for line in combined_cert_text.splitlines():
        line = line.strip(" •\t–—*-")
        if not line:
            continue
        # Match "Certified in X" or "X Certification" or "X – N Skill Badges"
        cert_match = re.search(
            r'[Cc]ertified\s+in\s+(.+?)(?:\s*[-–]\s*\d+|$)|'
            r'(.+?)\s+[Cc]ertification|'
            r'[Cc]ertified\s*:\s*(.+)',
            line
        )
        if cert_match:
            skill = next(g for g in cert_match.groups() if g)
            # Clean up trailing badge/count info
            skill = re.sub(r'\s*[-–]\s*\d+.*$', '', skill).strip()
            skill = re.sub(r'\s*\(.*?\)', '', skill).strip()
            if skill:
                certification_skills.append(skill)

    return {
        "links": {"github": github, "linkedin": linkedin},
        "skills": skills,
        "project_titles": project_titles,
        "certification_skills": certification_skills,
    }


if __name__ == "__main__":
    pdf_path = "resume.pdf"  # <-- change this
    print(json.dumps(extract_resume_data(pdf_path), indent=2))

Corpus size: 1615734
Sample: ['person has skills database administration database ms sql server ms sql server 2005 sql server sql server 2005 sql server 2008 sql server 2008 r2 sql server 2012 sql sql queries stored procedures clustering backups t sql virtualization r2 maintenance problem solving shipping database administration database ms sql server ms sql server 2005 sql server sql server 2005 sql server 2008 sql server 2008 r2 sql server 2012 sql sql queries stored procedures clustering backups t sql virtualization r2 maintenance problem solving shipping', 'person worked database administrator', 'person worked database administrator', 'person studied lead city university', 'person has ability installation and building server']


In [20]:
with open("corpus.txt", "w", encoding="utf-8") as f:
    for line in corpus:
        f.write(line + "\n")

print("✅ corpus.txt CREATED")

✅ corpus.txt CREATED


In [21]:
import os

print("Exists:", os.path.exists("corpus.txt"))
print("Working dir:", os.getcwd())
print("Files:", os.listdir())

Exists: True
Working dir: /content
Files: ['.config', '03_education.csv', '06_skills.csv', '01_people.csv', '05_person_skills.csv', 'corpus.txt', '02_abilities.csv', '04_experience.csv', 'sample_data']


In [22]:
unique_skills = list(set(skills_list))

if len(unique_skills) > 0:
    sentences.append("person has skills " + " ".join(unique_skills))

In [24]:
!pip install gensim
from gensim.models import Word2Vec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 69.1 MB/s eta 0:00:00


In [25]:
sentences = []

with open("corpus.txt", "r", encoding="utf-8") as f:
    for line in f:
        tokens = line.strip().split()
        if len(tokens) > 0:
            sentences.append(tokens)

print("Total sentences:", len(sentences))
print("Sample:", sentences[0][:10])

Total sentences: 1615734
Sample: ['person', 'has', 'skills', 'database', 'administration', 'database', 'ms', 'sql', 'server', 'ms']


In [26]:
from gensim.models.phrases import Phrases, Phraser

In [27]:
phrases = Phrases(sentences, min_count=2, threshold=5)
bigram = Phraser(phrases)

sentences = [bigram[sent] for sent in sentences]

In [29]:
!pip install tqdm

In [30]:
from gensim.models import Word2Vec
from tqdm import tqdm
import time

In [31]:
sentences_tokens = []

with open("corpus.txt", "r", encoding="utf-8") as f:
    for line in f:
        tokens = line.strip().split()
        if len(tokens) > 0:
            sentences_tokens.append(tokens)

print("Total sentences:", len(sentences_tokens))

Total sentences: 1615734


In [32]:
model = Word2Vec(
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1
)

model.build_vocab(sentences_tokens)

epochs = 10

print("\n🚀 Training Word2Vec Model...\n")

for epoch in tqdm(range(epochs), desc="Training Progress"):
    model.train(
        sentences_tokens,
        total_examples=model.corpus_count,
        epochs=1
    )
    time.sleep(0.2)  # just for visual effect

print("\n✅ Training Completed!")


🚀 Training Word2Vec Model...



Training Progress: 100%|██████████| 10/10 [09:08<00:00, 54.80s/it]


✅ Training Completed!


In [33]:
model.save("model.bin")
print("📦 model.bin saved successfully")

📦 model.bin saved successfully


In [34]:
from gensim.models import Word2Vec

# -----------------------------
# LOAD MODEL (SAFE)
# -----------------------------
try:
    model = Word2Vec.load("model.bin")
    print("✅ Word2Vec model loaded")
except:
    model = None
    print("⚠️ Model not found, similarity will return 0")

# -----------------------------
# CLEAN FUNCTION (IMPORTANT)
# -----------------------------
def clean_skill(skill):
    return skill.lower().strip()

# -----------------------------
# SKILL SIMILARITY FUNCTION
# -----------------------------
def skill_similarity(skill1, skill2):
    if model is None:
        return 0.0

    skill1 = clean_skill(skill1)
    skill2 = clean_skill(skill2)

    try:
        return float(model.wv.similarity(skill1, skill2))
    except KeyError:
        return 0.0

# -----------------------------
# GET EMBEDDING (OPTIONAL)
# -----------------------------
def get_embedding(skill):
    if model is None:
        return None

    skill = clean_skill(skill)

    try:
        return model.wv[skill]
    except KeyError:
        return None

✅ Word2Vec model loaded


In [36]:
print(skill_similarity("python", "flask"))
print(skill_similarity("sql", "database"))
print(skill_similarity("java", "photoshop"))

0.681631863117218
0.5356693863868713
0.3788093328475952


In [44]:
print(skill_similarity("flask", "kubernetes"))


0.5687413811683655


In [39]:
stopwords = {"person", "has", "skills", "worked", "studied", "ability"}

In [40]:
clean_sentences = []

stopwords = {"person", "has", "skills", "worked", "studied", "ability"}

with open("corpus.txt", "r", encoding="utf-8") as f:
    for line in f:
        tokens = line.strip().split()

        # remove stopwords
        tokens = [w for w in tokens if w not in stopwords]

        if len(tokens) > 1:
            clean_sentences.append(tokens)

print("Clean sentences:", len(clean_sentences))
print("Sample:", clean_sentences[0][:10])

Clean sentences: 1217748
Sample: ['database', 'administration', 'database', 'ms', 'sql', 'server', 'ms', 'sql', 'server', '2005']


In [41]:
from gensim.models.phrases import Phrases, Phraser

phrases = Phrases(clean_sentences, min_count=5, threshold=10)
bigram = Phraser(phrases)

clean_sentences = [bigram[s] for s in clean_sentences]